In [69]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------- Settings -----------------------------
MG1_MG2_settings = {
    "input_MG1_bracken_counts_dir": "/home/jpereira/Results/kraken_snake/MG1_20M/Data/merge_bracken/",
    "input_MG2_bracken_counts_dir": "/home/jpereira/Results/kraken_snake/MG2_20M/Data/merge_bracken/",
    "output_dir": "/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Data/merge_bracken/",
    "params_mean_abundance_filter": 0.001
}

load_settings = {"MG1_MG2": MG1_MG2_settings}

input_MG1_merge_bracken_dir = Path(load_settings["MG1_MG2"]["input_MG1_bracken_counts_dir"])
input_MG2_merge_bracken_dir = Path(load_settings["MG1_MG2"]["input_MG2_bracken_counts_dir"])
params_mean_abundance_filter = load_settings["MG1_MG2"]["params_mean_abundance_filter"]
output_dir = Path(load_settings["MG1_MG2"]["output_dir"])

output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------- Common taxa -----------------------------
# Get filenames (not full paths)
MG1_taxa = {d.name for d in input_MG1_merge_bracken_dir.iterdir() if d.is_dir()}
MG2_taxa = {d.name for d in input_MG2_merge_bracken_dir.iterdir() if d.is_dir()}

common_taxa = MG1_taxa.intersection(MG2_taxa)

# ----------------------------- Merge counts -----------------------------
taxa_l = []
taxa_num_before_l = []
taxa_num_after_l = []
for taxa in common_taxa:

    print(f"Processing taxa: {taxa}")
    
    MG1_taxa_dir = input_MG1_merge_bracken_dir / taxa
    MG2_taxa_dir = input_MG2_merge_bracken_dir / taxa
    
    MG1_files = {f.name: f for f in MG1_taxa_dir.iterdir() if f.is_file() and f.suffix == '.csv'}
    MG2_files = {f.name: f for f in MG2_taxa_dir.iterdir() if f.is_file() and f.suffix == '.csv'}
    
    common_files = set(MG1_files.keys()).intersection(set(MG2_files.keys()))
    
    output_taxa_dir = output_dir / taxa
    output_taxa_dir.mkdir(parents=True, exist_ok=True)
    

    for filename in common_files:
        if not filename.startswith("bracken_"):
            continue
        if filename.endswith("fraction_total_reads.csv"):
            continue
        
        print(f"  Merging file: {filename}")
        
        MG1_file_path = MG1_files[filename]
        MG2_file_path = MG2_files[filename]
        
        # Load data
        df_MG1 = pd.read_csv(MG1_file_path, sep=',')
        df_MG2 = pd.read_csv(MG2_file_path, sep=',')
        
        # Merge data on 'name' column
        merged_df = pd.merge(df_MG1, df_MG2, on='name', how='outer').fillna(0)
        merged_df.set_index('name', inplace=True)

        # Drop row with only zeros if exists
        if (merged_df == 0).all(axis=1).any():
            merged_df = merged_df[~(merged_df == 0).all(axis=1)]
        
        taxa_num_before = merged_df.shape[0]

        num = merged_df.select_dtypes('number')
        rel = num.div(num.sum(axis=0), axis=1)
        mean_rel_pct = rel.mean(axis=1) * 100

        vis_dir = output_dir / "abundance_histograms"
        vis_dir.mkdir(parents=True, exist_ok=True)

        mean_rel_pct = mean_rel_pct[mean_rel_pct > 0]
        log_mean_rel = mean_rel_pct.apply(np.log10)
        plt.figure(figsize=(10, 6))
        plt.hist(log_mean_rel, bins=100)
        plt.xlabel('Mean Relative Abundance (log10)')
        plt.ylabel('Frequency')
        plt.title(f'Histogram of Mean Relative Abundance ({taxa}) before filtering (log10)')
        plt.savefig(vis_dir / f'histogram_before_filtering_{taxa}.png')
        plt.close()

        mean_rel_pct = mean_rel_pct[mean_rel_pct > params_mean_abundance_filter]
        log_mean_rel = mean_rel_pct.apply(np.log10)
        plt.figure(figsize=(10, 6))
        plt.hist(log_mean_rel, bins=100)
        plt.xlabel('Mean Relative Abundance (log10)')
        plt.ylabel('Frequency')
        plt.title(f'Histogram of Mean Relative Abundance ({taxa}) after filtering (log10) > {params_mean_abundance_filter}')
        plt.savefig(vis_dir / f'histogram_after_filtering_{taxa}.png')
        plt.close()
        
        taxa_num_after = mean_rel_pct.shape[0]
        taxa_l.append(taxa)
        taxa_num_before_l.append(taxa_num_before)
        taxa_num_after_l.append(taxa_num_after)

        final_df = merged_df.loc[mean_rel_pct.index]
        final_df.to_csv(output_taxa_dir / f'merged_{taxa}_counts_filtered.csv', sep=',', index=True)

summary_df = pd.DataFrame({
    'Taxa': taxa_l,
    'Num_Taxa_Before_Filtering': taxa_num_before_l,
    'Num_Taxa_After_Filtering': taxa_num_after_l
})
summary_df.to_csv(output_dir / f'summary_num_taxa.tsv', sep='\t', index=False)



Processing taxa: S
  Merging file: bracken_merged_S_new_est_reads.csv
Processing taxa: O
  Merging file: bracken_merged_O_new_est_reads.csv
Processing taxa: C
  Merging file: bracken_merged_C_new_est_reads.csv
Processing taxa: G
  Merging file: bracken_merged_G_new_est_reads.csv
Processing taxa: F
  Merging file: bracken_merged_F_new_est_reads.csv
Processing taxa: P
  Merging file: bracken_merged_P_new_est_reads.csv


In [68]:
# import pandas as pd
# from pathlib import Path

# # ─── Settings ────────────────────────────────────────────────────────────────
# load_settings = "MT1_MT2_S" 
# if load_settings == "MG1_MG2_G":
#     input_metadata_tsv = Path('/home/jpereira/kraken_snake/metadata/metadata.MG1_MG2.csv')
#     input_bracken_counts_tsv = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/ClustDiv/bracken_combined_abundance.tsv')
#     input_bracken_counts_tsv = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Genera/Data/bracken_merged_G_new_est_reads.T.tsv')
#     params_select_categories_dict = {
#         'PR_R': {'Group': ['PR_R']},
#         'PR_E': {'Group': ['PR_E']},
#         'NS_R': {'Group': ['NS_R']},
#         'all_R': {'Group': ['PR_R', 'NS_R']}
#     }
#     params_samples_names = "LABID"
#     params_metadata_subtitle = False
#     output_subdivided_group_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera')
# elif load_settings == "MT1_MT2_S":
#     input_metadata_tsv = Path('/home/jpereira/kraken_snake/metadata/metadata.M1_M2.tsv')
#     input_bracken_counts_tsv = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/bracken_cow_rumen/S/bracken_merged_S_new_est_reads.csv')
#     params_select_categories_dict = {
#         'R1': {'diet_batch': ['R1']},
#         'E1': {'diet_batch': ['E1']},
#         'R2': {'diet_batch': ['R2']},
#         'all_R': {'diet_batch': ['R1', 'R2']}
#     }
#     params_samples_names = "mt_work_id"
#     params_counts_sep = ','
#     params_metadata_subtitle = False
#     params_transpose_counts = True
#     output_subdivided_group_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S')

# # ─── Prepare Output ──────────────────────────────────────────────────────────
# output_subdivided_group_dir.mkdir(exist_ok=True, parents=True)
# metadata_dir = output_subdivided_group_dir / 'metadata'
# counts_dir = output_subdivided_group_dir / 'counts'
# metadata_dir.mkdir(exist_ok=True)
# counts_dir.mkdir(exist_ok=True)

# # ─── Load Data ───────────────────────────────────────────────────────────────
# metadata_df = pd.read_csv(input_metadata_tsv, sep='\t', index_col=params_samples_names)
# counts_df = pd.read_csv(input_bracken_counts_tsv, sep=params_counts_sep, index_col=0)

# #if params_transpose_counts:
# #    counts_df = counts_df.T

# #counts_df



In [78]:
import pandas as pd
from pathlib import Path

# ─── Settings dictionaries ────────────────────────────────────────────────────
MG1_MG2_G_settings = {
    'input_metadata_tsv': Path('/home/jpereira/kraken_snake/metadata/metadata.MG1_MG2.csv'),
    # Note: in your original code this variable was assigned twice; the second path
    # overwrote the first. I'm preserving the final (effective) one here:
    'input_bracken_counts_tsv': Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Genera/Data/bracken_merged_G_new_est_reads.T.tsv'),
    'params_select_categories_dict': {
        'PR_R': {'Group': ['PR_R']},
        'PR_E': {'Group': ['PR_E']},
        'NS_R': {'Group': ['NS_R']},
        'all_R': {'Group': ['PR_R', 'NS_R']},
    },
    'params_samples_names': 'LABID',
    'params_metadata_subtitle': False,
    # Optional conveniences for consistent access across configs:
    'params_counts_sep': '\t',          # sensible default for .tsv
    'params_transpose_counts': False,   # not set in original; defaulting to False
    'output_subdivided_group_dir': Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera'),
}

MG1_MG2_settings = {
    'input_metadata_tsv': Path('/home/jpereira/kraken_snake/metadata/metadata.MG1_MG2.csv'),
    'input_bracken_counts_dir': Path('/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Data/merge_bracken'),
    'params_file_suffix': '_counts_filtered.csv',
    'params_select_categories_dict': {
        'PR_R': {'Group': ['PR_R']},
        'PR_E': {'Group': ['PR_E']},
        'NS_R': {'Group': ['NS_R']},
        'all_R': {'Group': ['PR_R', 'NS_R']},
    },
    'params_samples_names': 'LABID',
    'params_metadata_subtitle': False,
    # Optional conveniences for consistent access across configs:
    'params_counts_sep': ',',         
    'params_transpose_counts': True, 
    'output_subdivided_group_dir': Path('/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/'),
}


MT1_MT2_S_settings = {
    'input_metadata_tsv': Path('/home/jpereira/kraken_snake/metadata/metadata.M1_M2.tsv'),
    'input_bracken_counts_tsv': Path('/home/jpereira/Results/kraken_snake/MT1_MT2/bracken_cow_rumen/S/bracken_merged_S_new_est_reads.csv'),
    'params_select_categories_dict': {
        'R1': {'diet_batch': ['R1']},
        'E1': {'diet_batch': ['E1']},
        'R2': {'diet_batch': ['R2']},
        'all_R': {'diet_batch': ['R1', 'R2']},
    },
    'params_samples_names': 'mt_work_id',
    'params_counts_sep': ',',           # CSV in original
    'params_metadata_subtitle': False,
    'params_transpose_counts': True,
    'output_subdivided_group_dir': Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S'),
}

MT1_MT2_G_settings = {
    'input_metadata_tsv': Path('/home/jpereira/kraken_snake/metadata/metadata.M1_M2.tsv'),
    'input_bracken_counts_tsv': Path('/home/jpereira/Results/kraken_snake/MT1_MT2/bracken_cow_rumen/G/bracken_merged_G_new_est_reads.csv'),
    'params_select_categories_dict': {
        'R1': {'diet_batch': ['R1']},
        'E1': {'diet_batch': ['E1']},
        'R2': {'diet_batch': ['R2']},
        'all_R': {'diet_batch': ['R1', 'R2']},
    },
    'params_samples_names': 'mt_work_id',
    'params_counts_sep': ',',           # CSV in original
    'params_metadata_subtitle': False,
    'params_transpose_counts': True,
    'output_subdivided_group_dir': Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/G'),
}


# ─── Registry (choose with `work_name`) ───────────────────────────────────────
load_settings = {
    'MG1_MG2': MG1_MG2_settings,
    'MT1_MT2_S': MT1_MT2_S_settings,
    'MT1_MT2_G': MT1_MT2_G_settings,
}

# Pick the active configuration
work_name = 'MG1_MG2' 

# ─── Unpack convenience variables ─────────────────────────────────────────────
cfg = load_settings[work_name]

input_metadata_tsv         = cfg['input_metadata_tsv']
input_bracken_counts_dir   = cfg['input_bracken_counts_dir']
params_file_suffix        = cfg['params_file_suffix']
params_select_categories_dict   = cfg['params_select_categories_dict']
params_samples_names       = cfg['params_samples_names']
params_counts_sep          = cfg.get('params_counts_sep', '\t')
params_metadata_subtitle   = cfg['params_metadata_subtitle']
params_transpose_counts    = cfg.get('params_transpose_counts', False)
output_subdivided_group_dir= cfg['output_subdivided_group_dir']



# ─── Load Data ───────────────────────────────────────────────────────────────
metadata_df = pd.read_csv(input_metadata_tsv, sep='\t', index_col=params_samples_names)
for taxa_dir in input_bracken_counts_dir.iterdir():
    if not taxa_dir.is_dir():
        continue
    taxa = taxa_dir.name
    print(f"PROCESSING TAXA: {taxa}")

    # ─── Prepare Output ──────────────────────────────────────────────────────────
    output_subdivided_group_dir.mkdir(exist_ok=True, parents=True)
    metadata_dir = output_subdivided_group_dir / 'metadata' / taxa
    counts_dir = output_subdivided_group_dir / 'counts' / taxa
    metadata_dir.mkdir(exist_ok=True, parents=True)
    counts_dir.mkdir(exist_ok=True, parents=True)

    
    for filename in taxa_dir.iterdir():
        if not filename.is_file():
            continue
        if not filename.name.endswith(params_file_suffix):
            continue
        print(f"  Loading counts file: {filename.name}")
        counts_df = pd.read_csv(filename, sep=params_counts_sep, index_col=0)

        if params_transpose_counts:
            counts_df = counts_df.T

        counts_df.index.name = params_samples_names

        # ─── Remove Subtitle Row (if applicable) ─────────────────────────────────────
        if params_metadata_subtitle:
            metadata_df = metadata_df.iloc[1:, :]

        # ─── Validate Columns and Categories ─────────────────────────────────────────
        for group_name, filter_dict in params_select_categories_dict.items():
            column = list(filter_dict.keys())[0]
            values = filter_dict[column]

            if column not in metadata_df.columns:
                raise ValueError(f"     ❌ Column '{column}' (used in '{group_name}') not found in metadata.")

            missing_categories = set(values) - set(metadata_df[column].unique())
            if missing_categories:
                raise ValueError(
                    f"     ❌ Group '{group_name}' uses values not present in column '{column}': {missing_categories}"
                )

        print("     ✅ All column names and category values are valid.\n")

        # ─── Filter and Export ───────────────────────────────────────────────────────
        for group_name, filter_dict in params_select_categories_dict.items():
            print(f"\n🔹 Processing group: {group_name}")

            # Define output paths
            metadata_group_dir = metadata_dir / group_name
            counts_group_dir = counts_dir / group_name
            metadata_group_dir.mkdir(exist_ok=True, parents=True)
            counts_group_dir.mkdir(exist_ok=True, parents=True)

            metadata_group_tsv = metadata_group_dir / f"metadata.tsv"
            counts_group_tsv = counts_group_dir / f"counts.tsv"

            # Extract column name and values
            column = list(filter_dict.keys())[0]
            values = filter_dict[column]

            # Filter metadata
            m_df = metadata_df[metadata_df[column].isin(values)]

            # Check for missing samples
            missing_samples = set(m_df.index) - set(counts_df.index)
            if missing_samples:
                print(f"     ⚠️ Warning: {len(missing_samples)} sample(s) from metadata not found in counts_df:")
                for sample in sorted(missing_samples):
                    print(f"         - {sample}")

            # Filter counts using safe intersection
            shared_samples = m_df.index.intersection(counts_df.index)
            c_df = counts_df.loc[shared_samples]

            # Save filtered data
            m_df.loc[shared_samples].to_csv(metadata_group_tsv, sep="\t", index=True)
            c_df.to_csv(counts_group_tsv, sep="\t", index=True)

            print(f"     ✅ Exported: {len(shared_samples)} sample(s) to:")
            print(f"       - Metadata: {metadata_group_tsv}")
            print(f"       - Counts  : {counts_group_tsv}")


PROCESSING TAXA: C
  Loading counts file: merged_C_counts_filtered.csv
     ✅ All column names and category values are valid.


🔹 Processing group: PR_R
     ✅ Exported: 10 sample(s) to:
       - Metadata: /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/metadata/C/PR_R/metadata.tsv
       - Counts  : /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/counts/C/PR_R/counts.tsv

🔹 Processing group: PR_E
     ✅ Exported: 10 sample(s) to:
       - Metadata: /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/metadata/C/PR_E/metadata.tsv
       - Counts  : /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/counts/C/PR_E/counts.tsv

🔹 Processing group: NS_R
     ✅ Exported: 10 sample(s) to:
       - Metadata: /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/metadata/C/NS_R/metadata.tsv
       - Counts  : /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/counts/C/NS_R/counts.tsv

🔹 Processing group: all_R
     ✅ Exported: 20 sample(s) to:
       - Metadata: /

In [89]:
# ── Config registry refactor ──────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
from inmoose.pycombat import pycombat_norm, pycombat_seq
from inmoose.cohort_qc import CohortMetric, QCReport

# ── Individual configs (kept faithful to your originals) ──────────────────────
MG1_MG2_G_settings = {
    "input_metadata_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera/metadata"),
    "input_bracken_counts_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera/counts"),
    "params_batch_dict": {"NS_R": "Group", "PR_E": "Group", "PR_R": "Group", "all_R": "Group"},
    "params_effect_dict": {"NS_R": "Efficiency", "PR_E": "Efficiency", "PR_R": "Efficiency", "all_R": "Efficiency"},
    "params_normalization_methods": ["combat_seq"],
    "output_normalization_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2/Normalization/Genera/"),
    "params_index_cols": "LABID",
}

MT1_MT2_S_settings = {
    "input_metadata_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S/metadata"),
    "input_bracken_counts_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S/counts/"),
    "params_batch_dict": {"R1": "diet_batch", "E1": "diet_batch", "R2": "diet_batch", "all_R": "diet_batch"},
    "params_effect_dict": {"R1": "Pheno_RFI", "E1": "Pheno_RFI", "R2": "Pheno_RFI", "all_R": "Pheno_RFI"},
    "params_normalization_methods": ["combat_norm", "combat_seq"],
    "output_normalization_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Normalization/S/"),
    "params_index_cols": "mt_work_id",
}

MT1_MT2_G_settings = {
    "input_metadata_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/G/metadata"),
    "input_bracken_counts_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/G/counts/"),
    "params_batch_dict": {"R1": "diet_batch", "E1": "diet_batch", "R2": "diet_batch", "all_R": "diet_batch"},
    "params_effect_dict": {"R1": "Pheno_RFI", "E1": "Pheno_RFI", "R2": "Pheno_RFI", "all_R": "Pheno_RFI"},
    "params_normalization_methods": ["combat_norm", "combat_seq"],
    "output_normalization_dir": Path("/home/jpereira/Results/kraken_snake/MT1_MT2/Normalization/G/"),
    "params_index_cols": "mt_work_id",
}




# This one already followed your desired pattern; adding a couple of optional keys for consistency
MG1_MG2_settings = {
    "input_metadata_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/metadata/"),
    "input_bracken_counts_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Groups/counts"),
    "params_batch_dict": {"NS_R": "Group", "PR_E": "Group", "PR_R": "Group", "all_R": "Group"},
    "params_effect_dict": {"NS_R": "Efficiency", "PR_E": "Efficiency", "PR_R": "Efficiency", "all_R": "Efficiency"},
    "params_normalization_methods": ["combat_seq"],
    "params_index_cols": "LABID",
    "params_counts_sep": ",",
    "output_normalization_dir": Path("/home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/"),
}

# ── Registry (choose with `work_name`) ────────────────────────────────────────
load_settings = {
    "MG1_MG2_G": MG1_MG2_G_settings,
    "MT1_MT2_S": MT1_MT2_S_settings,
    "MT1_MT2_G": MT1_MT2_G_settings,
    "MG1_MG2": MG1_MG2_settings,  # the merge/selection flow
}

# ── Pick the active configuration ─────────────────────────────────────────────
# Examples: "MG1_MG2_G", "MT1_MT2_S", "MT1_MT2_G", "MG1_MG2"
work_name = "MG1_MG2"

# ── Unpack convenience variables (works for both “normalization” and “merge” flows) ─
cfg = load_settings[work_name]

# Common keys
input_bracken_counts_dir   = cfg.get("input_bracken_counts_dir")
params_counts_sep          = cfg.get("params_counts_sep", "\t")
params_transpose_counts    = cfg.get("params_transpose_counts", False)

# For normalization (MG1_MG2_G / MT1_MT2_* flows)
input_metadata_dir         = cfg.get("input_metadata_dir")
params_batch_dict          = cfg.get("params_batch_dict")
params_effect_dict         = cfg.get("params_effect_dict")
params_normalization_methods = cfg.get("params_normalization_methods", [])
params_index_cols          = cfg.get("params_index_cols")
output_normalization_dir   = cfg.get("output_normalization_dir")

# From here on, write workflow-agnostic code using the variables above.
# Example:
# - if work_name == "MG1_MG2": do merge/selection using input_metadata_tsv & params_select_categories_dict
# - else: do ComBat normalization using input_metadata_dir & params_*_dict

In [90]:
from pathlib import Path
import pandas as pd
from inmoose.pycombat import pycombat_norm, pycombat_seq
from inmoose.cohort_qc import CohortMetric, QCReport

# #test_mode = True
# load_settings
# if load_settings == 'MG1_MG2_G':
#     input_metadata_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera/metadata')#/metadata.NS_R.tsv')
#     input_bracken_counts_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/Genera/counts')#/counts.NS_R.tsv')
#     params_batch_dict = {'NS_R' : 'Group', 'PR_E' : 'Group', 'PR_R' : 'Group', 'all_R' : 'Group'}
#     params_effect_dict = {'NS_R' : 'Efficiency', 'PR_E' : 'Efficiency', 'PR_R' : 'Efficiency', 'all_R' : 'Efficiency'}
#     params_normalization_methods = ['combat_norm', 'combat_seq']
#     output_normalization_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Normalization/Genera/Efficiency.cov')
#     params_index_cols = 'LABID'
# elif load_settings == "MT1_MT2_S":
#     input_metadata_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S/metadata')
#     input_bracken_counts_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/S/counts/')
#     params_batch_dict = {
#         'R1': 'diet_batch',
#         'E1': 'diet_batch',
#         'R2': 'diet_batch',
#         'all_R': 'diet_batch'
#         }
#     params_effect_dict = {
#     'R1': 'Pheno_RFI',
#     'E1': 'Pheno_RFI',
#     'R2': 'Pheno_RFI',
#     'all_R': 'Pheno_RFI'
#     }
#     params_normalization_methods = ['combat_norm', 'combat_seq']
#     output_normalization_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Normalization/S/')
#     params_index_cols = 'mt_work_id'
# elif load_settings == "MT1_MT2_G":
#     input_metadata_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/G/metadata')
#     input_bracken_counts_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Groups/G/counts/')
#     params_batch_dict = {
#         'R1': 'diet_batch',
#         'E1': 'diet_batch',
#         'R2': 'diet_batch',
#         'all_R': 'diet_batch'
#         }
#     params_effect_dict = {
#     'R1': 'Pheno_RFI',
#     'E1': 'Pheno_RFI',
#     'R2': 'Pheno_RFI',
#     'all_R': 'Pheno_RFI'
#     }
#     params_normalization_methods = ['combat_norm', 'combat_seq']
#     output_normalization_dir = Path('/home/jpereira/Results/kraken_snake/MT1_MT2/Normalization/G/')
#     params_index_cols = 'mt_work_id'

counts_taxa_dirs = [d.name for d in input_bracken_counts_dir.iterdir() if d.is_dir()]
metadata_taxa_dirs = [d.name for d in input_metadata_dir.iterdir() if d.is_dir()]

common_taxa_dirs = set(counts_taxa_dirs).intersection(set(metadata_taxa_dirs))

for taxa in common_taxa_dirs:
    for group_name, batch_col in params_batch_dict.items():

        # 1) Read metadata & counts ------------------------------------------------

        meta_tsv = input_metadata_dir / taxa / group_name / f"metadata.tsv"
        counts_tsv = input_bracken_counts_dir / taxa / group_name / f"counts.tsv"
        if not meta_tsv.exists() or not counts_tsv.exists():
            print(f"[{group_name}] Skipping {taxa}: missing metadata or counts file.")
            continue

        meta_df   = pd.read_csv(meta_tsv, sep="\t", index_col=params_index_cols)
        counts_df = pd.read_csv(counts_tsv, sep="\t", index_col=0)

        # 2) Align samples ---------------------------------------------------------
        if set(meta_df.index).issubset(counts_df.columns):
            counts = counts_df[meta_df.index]
        elif set(meta_df.index).issubset(counts_df.index):
            counts = counts_df.loc[meta_df.index].T
        else:
            raise ValueError(f"[{group_name}] Skipping {taxa}: Sample IDs in metadata and counts do not match.")

        batch_vector   = meta_df.loc[counts.columns, batch_col].to_numpy()
        effect_vector  = meta_df.loc[counts.columns, params_effect_dict[group_name]].to_numpy()
        unique_batches = pd.unique(batch_vector)

        # 3) Make output dir -------------------------------------------------------
        out_dir = output_normalization_dir / taxa / group_name
        out_dir.mkdir(parents=True, exist_ok=True)

        if unique_batches.size < 2:
            print(f"[{group_name}] Skipping {taxa}: Only one batch ({unique_batches[0]!r}); "
                "skipping ComBat.")
            print(f"[{group_name}] Skipping {taxa}: Saving raw counts in {out_dir / f"counts.tsv"}")
            counts.to_csv(out_dir / f"counts.tsv",
                                sep="\t")
            continue
        
        print(f"[{group_name}] {taxa}: Detected {unique_batches.size} batches: {unique_batches}")

        # 4) ComBat log-normal -----------------------------------------------------
        if 'combat_norm' in params_normalization_methods:
            combat_norm_df = pycombat_norm(counts, batch=batch_vector, covar_mod=effect_vector)
            combat_norm_df.to_csv(out_dir / f"combat_norm.tsv", sep="\t")

            # --- PCA/QC report (colors by batch) ---
            cm = CohortMetric(
                clinical_df=meta_df[[batch_col]],   # you can add more covariates here
                batch_column=batch_col,
                data_expression_df=combat_norm_df,  # after correction
                data_expression_df_before=counts    # before correction (for comparison)
            )
            cm.process()

            QCReport(cm).save_report(output_normalization_dir / taxa / group_name / f"qc_report.combat_norm.html")  # opens with PCA plots colored by Group

        # 5) ComBat-Seq (counts) ---------------------------------------------------
        if 'combat_seq' in params_normalization_methods:
            combat_seq_df = pycombat_seq(counts, batch=batch_vector)
            combat_seq_df.to_csv(out_dir / f"combat_seq.tsv", sep="\t")
            
            # --- PCA/QC report (colors by batch) ---
            cm = CohortMetric(
                clinical_df=meta_df[[batch_col]],          # you can add more covariates here
                batch_column=batch_col,
                data_expression_df=combat_seq_df,  # after correction
                data_expression_df_before=counts      # before correction (for comparison)
            )
            cm.process()

            QCReport(cm).save_report(output_normalization_dir / taxa / group_name / f"qc_report.combat_seq.html")  # opens with PCA plots colored by Group

[NS_R] Skipping S: Only one batch ('NS_R'); skipping ComBat.
[NS_R] Skipping S: Saving raw counts in /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/S/NS_R/counts.tsv
[PR_E] Skipping S: Only one batch ('PR_E'); skipping ComBat.
[PR_E] Skipping S: Saving raw counts in /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/S/PR_E/counts.tsv
[PR_R] Skipping S: Only one batch ('PR_R'); skipping ComBat.
[PR_R] Skipping S: Saving raw counts in /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/S/PR_R/counts.tsv
[all_R] S: Detected 2 batches: ['PR_R' 'NS_R']
[NS_R] Skipping O: Only one batch ('NS_R'); skipping ComBat.
[NS_R] Skipping O: Saving raw counts in /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/O/NS_R/counts.tsv
[PR_E] Skipping O: Only one batch ('PR_E'); skipping ComBat.
[PR_E] Skipping O: Saving raw counts in /home/jpereira/Results/kraken_snake/MG1_MG2_GTDB/Normalization/O/PR_E/counts.tsv
[PR_R] Skipping O: Only one batch ('PR_R');

In [3]:
pd.read_csv('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/metadata/metadata.NS_R.tsv', sep='\t')

,LABID,Prueba,Dieta,Efficiency,Group
0,NS001_MG,26.0,R,High_Efficiency,NS_R
1,NS002_MG,26.0,R,High_Efficiency,NS_R
2,NS003_MG,26.0,R,High_Efficiency,NS_R
3,NS004_MG,26.0,R,High_Efficiency,NS_R
4,NS005_MG,26.0,R,High_Efficiency,NS_R
5,NS006_MG,26.0,R,Low_Efficiency,NS_R
6,NS007_MG,26.0,R,Low_Efficiency,NS_R
7,NS008_MG,26.0,R,Low_Efficiency,NS_R
8,NS009_MG,26.0,R,Low_Efficiency,NS_R
9,NS010_MG,26.0,R,Low_Efficiency,NS_R


In [2]:
batch_vector

array(['PR_R', 'PR_R', 'PR_R', 'PR_R', 'PR_R', 'PR_R', 'PR_R', 'PR_R',
       'PR_R', 'PR_R', 'NS_R', 'NS_R', 'NS_R', 'NS_R', 'NS_R', 'NS_R',
       'NS_R', 'NS_R', 'NS_R', 'NS_R'], dtype=object)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from inmoose.pycombat import pycombat_norm, pycombat_seq
from inmoose.cohort_qc import CohortMetric, QCReport
from inmoose.deseq2 import deseq2_cpp

# --- NEW: imports for DA ---
from scipy.stats import mannwhitneyu
from statsmodels.api import GLM, add_constant
from statsmodels.genmod.families import NegativeBinomial
from statsmodels.stats.multitest import multipletests

# --------------- CONFIG ----------------
test_mode = True
if test_mode:
    input_metadata_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/metadata')
    input_bracken_counts_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/counts/')
    params_batch_dict = {'NS_R' : 'Group', 'PR_E' : 'Group', 'PR_R' : 'Group', 'all_R' : 'Group'}
    params_normalization_methods = ['combat_norm', 'combat_seq']
    output_normalization_dir = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Normalization/')
    params_index_cols = 'LABID'
    # --- NEW: DA target (edit these) ---
    params_da_col = 'Efficiency'       # column in metadata with two classes
    params_da_baseline = 'Low_Efficiency'    # reference level for effect signs

# ---------- Helpers ----------
def clr_transform(df, pseudocount=1.0):
    # df: features x samples (like you save them)
    X = df.copy().astype(float) + pseudocount
    gm = np.exp(np.log(X).mean(axis=0))  # geometric mean per sample
    return np.log(X / gm)

def wilcoxon_da(expr_df, meta, group_col, baseline):
    """
    expr_df: features x samples (numeric), e.g., CLR data
    meta: dataframe indexed by sample with group_col
    """
    s = meta[group_col].astype(str)
    groups = s.unique()
    if len(groups) != 2:
        raise ValueError(f"{group_col} must have exactly 2 levels for Wilcoxon.")
    case = [g for g in groups if g != baseline][0]
    case_idx = s[s == case].index
    base_idx = s[s == baseline].index

    results = []
    for feat, row in expr_df.iterrows():
        x = row[case_idx].dropna().values
        y = row[base_idx].dropna().values
        if len(x) < 2 or len(y) < 2:
            p = np.nan
            l2fc = np.nan
        else:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
            l2fc = np.nanmean(x) - np.nanmean(y)
        results.append((feat, l2fc, p))

    res_df = pd.DataFrame(results, columns=['feature', 'effect_CLR_diff', 'p'])

    # --- FIX: handle NaNs manually ---
    mask = ~res_df['p'].isna()
    res_df['q'] = np.nan
    if mask.any():
        res_df.loc[mask, 'q'] = multipletests(res_df.loc[mask, 'p'], method='fdr_bh')[1]

    return res_df.sort_values('q')

def nb_glm_da(counts_df, meta, group_col, baseline):
    """
    counts_df: features x samples (features as rows, samples as columns), after ComBat-Seq
    """
    s = meta[group_col].astype('category')

    # Ensure baseline exists and is reference
    if baseline not in list(s.cat.categories):
        raise ValueError(f"Baseline level '{baseline}' not found in '{group_col}'. Found: {list(s.cat.categories)}")
    s = s.cat.reorder_categories([baseline] + [c for c in s.cat.categories if c != baseline])
    if s.nunique() != 2:
        raise ValueError(f"{group_col} must have exactly 2 levels for NB-GLM. Found {s.nunique()}")

    # Design: intercept + one dummy for 'case' vs baseline
    design = pd.get_dummies(s, drop_first=True)
    design = add_constant(design, has_constant='add')
    design.index.name = counts_df.columns.name

    results = []
    X = design.values
    for feat, row in counts_df.iterrows():
        y = row.loc[design.index].astype(float).values
        # Guard against all-zeros / no variance
        if np.all(y == 0) or np.nanstd(y) == 0:
            beta, p = np.nan, np.nan
        else:
            try:
                model = GLM(y, X, family=NegativeBinomial())
                fit = model.fit()
                beta = fit.params[1]   # coefficient for case-vs-baseline
                p    = fit.pvalues[1]
            except Exception:
                beta, p = np.nan, np.nan
        results.append((feat, beta, p))

    res_df = pd.DataFrame(results, columns=['feature', 'beta_case_vs_baseline', 'p'])

    # --- FIX: handle NaNs manually ---
    mask = ~res_df['p'].isna()
    res_df['q'] = np.nan
    if mask.any():
        res_df.loc[mask, 'q'] = multipletests(res_df.loc[mask, 'p'], method='fdr_bh')[1]

    return res_df.sort_values('q', na_position='last')

def deseq2_da_inmoose(
    counts_df: pd.DataFrame,
    meta: pd.DataFrame,
    condition_col: str,
    baseline_level: str,
    batch_col: str | None = None,
    lfc_shrink: bool = False,
    fit_type: str = "mean",          # <-- use 'mean' to avoid the parametric/local issue
    min_total_count: int = 25,       # <-- simple prefilter helps dispersion fit
    min_samples_nonzero: int = 4,    # <-- another light filter
):
    """
    Run DESeq2 via InMoose on counts_df (features x samples).
    Returns a pandas DataFrame with DESeq2 results.
    """

    # --- Align samples ---
    samples = counts_df.columns
    coldata = meta.loc[samples, [c for c in [batch_col, condition_col] if c is not None]].copy()

    # --- Ensure condition baseline first ---
    cond_levels = list(pd.Index(coldata[condition_col]).astype(str).unique())
    other_levels = [lvl for lvl in cond_levels if lvl != baseline_level]
    coldata[condition_col] = pd.Categorical(
        coldata[condition_col].astype(str),
        categories=[baseline_level] + other_levels,
        ordered=True
    )
    if batch_col:
        coldata[batch_col] = pd.Categorical(coldata[batch_col].astype(str))

    # --- Light prefiltering (keeps stats stable) ---
    X = counts_df.loc[:, samples].copy()
    keep = (X.sum(axis=1) >= min_total_count) & ((X > 0).sum(axis=1) >= min_samples_nonzero)
    X = X.loc[keep]
    if X.empty:
        raise ValueError("All features filtered out before DESeq2. Lower the filtering thresholds.")

    # --- Build design with condition last ---
    design_terms = []
    if batch_col:
        design_terms.append(batch_col)
    design_terms.append(condition_col)
    design = "~ " + " + ".join(design_terms)

    # --- Run DESeq2 via InMoose ---
    from inmoose.deseq2 import DESeqDataSet, DESeq
    dds = DESeqDataSet(countData=X.T, clinicalData=coldata, design=design)

    # Prefer fitType='mean' because 'local' isn’t implemented in InMoose
    try:
        dds = DESeq(dds, fitType=fit_type)
    except TypeError:
        # Some versions don’t expose fitType in the call; try setting attribute then running
        try:
            setattr(dds, "fitType", fit_type)
            dds = DESeq(dds)
        except Exception as e:
            raise RuntimeError(f"DESeq run failed even after setting fitType='{fit_type}': {repr(e)}")

    # Determine case vs baseline
    case_level = next(l for l in coldata[condition_col].cat.categories if l != baseline_level)
    coef_name = f"{condition_col}_{case_level}_vs_{baseline_level}"

    # Pull results (try contrast first, then name=)
    try:
        res = dds.results(contrast=[condition_col, case_level, baseline_level])
    except Exception:
        res = dds.results(name=coef_name)

    if lfc_shrink:
        try:
            res = dds.lfcShrink(coef=coef_name, type="apeglm")
        except Exception as e:
            print(f"[DESeq2] LFC shrinkage failed ({repr(e)}); returning unshrunken results.")

    if "feature" not in res.columns:
        res = res.reset_index().rename(columns={"index": "feature"})
    return res


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_volcano(
    df,
    effect_col,
    pval_col,
    out_file,
    title=None,
    alpha=0.05,
    lfc_thresh=1.0,
    feature_col="feature",
    max_labels=20,
    figsize=(18, 12),
    jitter_amount=0.05,  # controls how far annotations can be randomly shifted
):
    """
    Volcano plot (bigger, with annotations and jitter).
    - Uses random jitter in annotation positions to reduce overlap.
    - Labels: all significant points + top hits by -log10(p), up to `max_labels`.
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import math

    if df is None or (hasattr(df, "empty") and df.empty):
        print(f"[Volcano] No data for {title}; skipping.")
        return

    df = pd.DataFrame(df).copy()

    # Check required columns
    for col in [effect_col, pval_col]:
        if col not in df.columns:
            print(f"[Volcano] Missing required column '{col}' for {title}; skipping.")
            return

    # Keep needed columns, coerce numeric, drop NaNs
    keep_cols = [effect_col, pval_col] + ([feature_col] if feature_col in df.columns else [])
    df = df[keep_cols].apply(pd.to_numeric, errors="ignore")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[effect_col, pval_col])
    if df.empty:
        print(f"[Volcano] No valid rows for {title}; skipping.")
        return

    # Compute -log10(p)
    eps = np.finfo(float).tiny
    df["_neglog10p"] = -np.log10(np.clip(pd.to_numeric(df[pval_col], errors="coerce"), eps, None))

    # Significance mask
    sig_mask = (df[pval_col] < alpha) & (df[effect_col].abs() >= lfc_thresh)

    # Figure
    plt.figure(figsize=figsize)

    # Background
    plt.scatter(df[effect_col], df["_neglog10p"], s=12, alpha=0.35)

    # Foreground (significant)
    if sig_mask.any():
        plt.scatter(df.loc[sig_mask, effect_col], df.loc[sig_mask, "_neglog10p"], s=30, alpha=0.9)

    # Threshold lines
    plt.axvline(x=lfc_thresh,  linestyle="--", linewidth=1)
    plt.axvline(x=-lfc_thresh, linestyle="--", linewidth=1)
    plt.axhline(y=-math.log10(alpha), linestyle="--", linewidth=1)

    # Labels: annotate all significant + top remaining
    to_label = pd.Index([])
    if sig_mask.any():
        to_label = df.index[sig_mask]

    remaining = max(0, max_labels - len(to_label))
    if remaining > 0:
        top_extra = df.loc[~df.index.isin(to_label)].nlargest(remaining, "_neglog10p").index
        to_label = to_label.append(top_extra)

    # Annotate with jitter
    if feature_col in df.columns and len(to_label) > 0:
        rng = np.random.default_rng()
        for i in to_label:
            x = df.at[i, effect_col]
            y = df.at[i, "_neglog10p"]
            label = str(df.at[i, feature_col])

            # base offset
            dx = 0.02 if (hash(i) % 2 == 0) else -0.02
            dy = 0.2

            # add small random jitter
            dx += rng.uniform(-jitter_amount, jitter_amount)
            dy += rng.uniform(-jitter_amount, jitter_amount)

            plt.annotate(
                label,
                xy=(x, y),
                xytext=(x + dx, y + dy),
                textcoords="data",
                fontsize=6,
                arrowprops=dict(arrowstyle="-", lw=0.5),
            )

    plt.xlabel(effect_col)
    plt.ylabel(f"-log10({pval_col})")
    plt.title(title or "Volcano plot")
    plt.tight_layout()
    plt.savefig(out_file, dpi=300)
    plt.close()



from statsmodels.stats.multitest import multipletests

def tidy_deseq2_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Put index into a column if needed
    if 'feature' not in out.columns:
        out = out.reset_index().rename(columns={'index': 'feature'})

    # Normalize column names seen in different outputs
    rename_map = {
        'adj_pvalue': 'padj',
        'adj.P.Val':  'padj',
        'base_mean':  'baseMean',
        'log2fc':     'log2FoldChange',
        'SE':         'lfcSE',
        'statistic':  'stat',
        'p_value':    'pvalue',
        'pval':       'pvalue',
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    # Ensure lfcSE exists even if the backend didn't return it
    if 'lfcSE' not in out.columns:
        out['lfcSE'] = np.nan

    # Ensure padj exists; compute from pvalue if missing
    if 'padj' not in out.columns:
        out['padj'] = np.nan
        if 'pvalue' in out.columns:
            mask = ~out['pvalue'].isna()
            if mask.any():
                out.loc[mask, 'padj'] = multipletests(out.loc[mask, 'pvalue'], method='fdr_bh')[1]

    # Make numeric where relevant
    for c in ['baseMean','log2FoldChange','lfcSE','stat','pvalue','padj']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')

    return out




In [ ]:
# --------------- MAIN ------------------
for group_name, batch_col in params_batch_dict.items():

    # 1) Read metadata & counts ------------------------------------------------
    meta_df   = pd.read_csv(input_metadata_dir / f"metadata.{group_name}.tsv",
                            sep="\t", index_col=params_index_cols)
    counts_df = pd.read_csv(input_bracken_counts_dir / f"counts.{group_name}.tsv",
                            sep="\t", index_col=0)

    # 2) Align samples ---------------------------------------------------------
    if set(meta_df.index).issubset(counts_df.columns):
        counts = counts_df[meta_df.index]
    elif set(meta_df.index).issubset(counts_df.index):
        counts = counts_df.loc[meta_df.index].T
    else:
        raise ValueError("Sample IDs in metadata and counts do not match.")

    batch_vector = meta_df.loc[counts.columns, batch_col].to_numpy()
    unique_batches = pd.unique(batch_vector)
    has_batches = unique_batches.size >= 2
    if not has_batches:
        print(f"[{group_name}] Only one batch ({unique_batches[0]!r}); skipping ComBat but continuing with DA.")

    # 3) Make output dir -------------------------------------------------------
    out_dir = output_normalization_dir / group_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Prepare vars for DA
    combat_norm_df = None
    combat_seq_df  = None

    # 4) ComBat log-normal -----------------------------------------------------
    if has_batches and 'combat_norm' in params_normalization_methods:
        combat_norm_df = pycombat_norm(counts, batch=batch_vector)
        combat_norm_df.to_csv(out_dir / f"combat_norm.{group_name}.tsv", sep="\t")

        # PCA/QC
        cm = CohortMetric(
            clinical_df=meta_df.loc[counts.columns, [batch_col]],
            batch_column=batch_col,
            data_expression_df=combat_norm_df,
            data_expression_df_before=counts
        )
        cm.process()
        QCReport(cm).save_report(output_normalization_dir / f"qc_report.combat_norm.{group_name}.html")

    # 5) ComBat-Seq (counts) ---------------------------------------------------
    if has_batches and 'combat_seq' in params_normalization_methods:
        combat_seq_df = pycombat_seq(counts, batch=batch_vector,)
        combat_seq_df.to_csv(out_dir / f"combat_seq.{group_name}.tsv", sep="\t")

        # PCA/QC
        cm = CohortMetric(
            clinical_df=meta_df.loc[counts.columns, [batch_col]],
            batch_column=batch_col,
            data_expression_df=combat_seq_df,
            data_expression_df_before=counts
        )
        cm.process()
        QCReport(cm).save_report(output_normalization_dir / f"qc_report.combat_seq.{group_name}.html")

    # ------------------ DIFFERENTIAL ABUNDANCE ------------------
    if params_da_col in meta_df.columns:
        # Pick sources
        da_source_counts = combat_seq_df if combat_seq_df is not None else counts
        da_source_log    = combat_norm_df if combat_norm_df is not None else clr_transform(counts)

    # 1) DESeq2 via inmoose ------------------------------------
    da_source_counts = combat_seq_df if combat_seq_df is not None else counts

    # DESeq2
    try:
        deseq2_res = deseq2_da_inmoose(
            da_source_counts, meta_df,
            condition_col=params_da_col,
            baseline_level=params_da_baseline,
            batch_col=batch_col,
            lfc_shrink=False,     # keep False to avoid lfcSE-related errors
            fit_type="mean"
        )
        deseq2_res = tidy_deseq2_table(deseq2_res)

        # Sort by padj if present, else pvalue
        sort_col = 'padj' if deseq2_res['padj'].notna().any() else 'pvalue'
        deseq2_res = deseq2_res.sort_values(sort_col, na_position='last')

        deseq2_res.to_csv(out_dir / f"DA_deseq2.{group_name}.tsv", sep="\t", index=False)

        # Volcano: prefer padj; fallback to pvalue if padj is all NaN
        pcol = sort_col
        plot_volcano(
            deseq2_res, 'log2FoldChange', pcol,
            out_dir / f"Volcano_deseq2.{group_name}.png",
            title=f"DESeq2 Volcano - {group_name}"
        )

    except Exception as e:
        print(f"[{group_name}] DESeq2 (InMoose) failed: {repr(e)}")


    # NB-GLM
    try:
        nb_res = nb_glm_da(
            da_source_counts,
            meta_df.loc[da_source_counts.columns],
            params_da_col,
            params_da_baseline
        )

        # Sort by q if present, else p
        sort_col = 'q' if 'q' in nb_res.columns and nb_res['q'].notna().any() else 'p'
        nb_res = nb_res.sort_values(sort_col, na_position='last')

        out_tsv = out_dir / f"DA_nbGLM.{group_name}.tsv"
        nb_res.to_csv(out_tsv, sep="\t", index=False)

        plot_volcano(
            nb_res, 'beta_case_vs_baseline', sort_col,
            out_dir / f"Volcano_nbGLM.{group_name}.png",
            title=f"NB-GLM Volcano - {group_name}"
        )

    except Exception as e:
        print(f"[{group_name}] NB-GLM failed ({e}); falling back to Wilcoxon on CPM.")


    # Wilcoxon CLR
    try:
        wilcoxon_res = wilcoxon_da(
            da_source_log,
            meta_df.loc[da_source_log.columns],
            params_da_col,
            params_da_baseline
        )

        # Sort by q if present, else p
        sort_col = 'q' if 'q' in wilcoxon_res.columns and wilcoxon_res['q'].notna().any() else 'p'
        wilcoxon_res = wilcoxon_res.sort_values(sort_col, na_position='last')

        out_tsv = out_dir / f"DA_wilcoxon_CLR.{group_name}.tsv"
        wilcoxon_res.to_csv(out_tsv, sep="\t", index=False)

        plot_volcano(
            wilcoxon_res, 'effect_CLR_diff', sort_col,
            out_dir / f"Volcano_wilcoxon_CLR.{group_name}.png",
            title=f"Wilcoxon CLR Volcano - {group_name}"
        )

    except Exception as e:
        print(f"[{group_name}] Wilcoxon CLR failed: {e}")



[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship


[NS_R] Only one batch ('NS_R'); skipping ComBat but continuing with DA.
[NS_R] DESeq2 (InMoose) failed: NotImplementedError()
[Volcano] No valid rows for NB-GLM Volcano - NS_R; skipping.


[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship


[PR_E] Only one batch ('PR_E'); skipping ComBat but continuing with DA.
[PR_E] DESeq2 (InMoose) failed: NotImplementedError()
[Volcano] No valid rows for NB-GLM Volcano - PR_E; skipping.


[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship


[PR_R] Only one batch ('PR_R'); skipping ComBat but continuing with DA.
[PR_R] DESeq2 (InMoose) failed: NotImplementedError()
[Volcano] No valid rows for NB-GLM Volcano - PR_R; skipping.


[INFO] Found 2 batches
[INFO] Adjusting for 0 covariate(s) or covariate level(s)
[INFO] Standardizing Data across genes.
[INFO] Fitting L/S model and finding priors.
[INFO] Finding parametric adjustments.
[INFO] Adjusting the Data
[INFO] Found 2 batches
[INFO] Adjusting for 0 covariate(s) or covariate level(s)
[INFO] Estimating dispersions
[INFO] Fitting the GLM model
[INFO] shrinkage off - using GLM estimates for parameters
[INFO] Adjusting the data
[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship


[all_R] DESeq2 (InMoose) failed: NotImplementedError()
[Volcano] No valid rows for NB-GLM Volcano - all_R; skipping.


In [59]:
# DESeq2

deseq2_res = deseq2_da_inmoose(
    counts, meta_df,
    condition_col=params_da_col,
    baseline_level=params_da_baseline,
    batch_col=batch_col,
    lfc_shrink=False,     # keep False to avoid lfcSE-related errors
    fit_type="mean"
)
deseq2_res = tidy_deseq2_table(deseq2_res).sort_values('padj')
deseq2_res.to_csv(out_dir / f"DA_deseq2.counts.{group_name}.tsv", sep="\t", index=False)
# Volcano: prefer padj; fallback to pvalue if padj is all NaN
pcol = 'padj' if deseq2_res['padj'].notna().any() else 'pvalue'
plot_volcano(deseq2_res, 'log2FoldChange', pcol,
            out_dir / f"Volcano_deseq2.counts.{group_name}.png",
            title=f"DESeq2 Volcano.counts - {group_name}")


print(str(out_dir / f"Volcano_deseq2.counts.{group_name}.png"))

[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship
[INFO] final dispersion estimates
[INFO] fitting model and testing


/home/jpereira/Results/kraken_snake/MG1_MG2/Normalization/all_R/Volcano_deseq2.counts.all_R.png


In [54]:
deseq2_res.sort_values('padj')

,feature,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
2037,Sodaliphilus sp900320245,3994.649339,-1.755323,0.347955,-5.044686,4.542664e-07,0.000934
851,MGYG000293263,5226.882283,0.965072,0.198005,4.873989,1.093672e-06,0.001124
2033,Sodaliphilus sp900318645,1036.017569,-1.170382,0.251551,-4.652659,3.276820e-06,0.001684
2057,Sodaliphilus sp902787185,2292.290291,-2.147034,0.460412,-4.663289,3.111946e-06,0.001684
892,MGYG000293609,1093.300506,-1.989307,0.467351,-4.256563,2.075937e-05,0.005335
...,...,...,...,...,...,...,...
2708,UMGS1820 sp902797885,214.017462,0.260716,0.186998,1.394218,1.632518e-01,NaN
2711,UMGS874 sp902766575,121.981071,0.073859,0.138496,0.533293,5.938311e-01,NaN
2714,Weimerbacter sp002480925,165.631541,0.029043,0.180416,0.160979,8.721097e-01,NaN
2715,Weimerbacter sp900315505,174.545705,0.120308,0.195066,0.616753,5.373974e-01,NaN


In [42]:
deseq2_res = deseq2_da_inmoose(
da_source_counts, meta_df,
condition_col=params_da_col,
baseline_level=params_da_baseline,
batch_col=batch_col,
lfc_shrink=False,     # keep False to avoid lfcSE-related errors
fit_type="mean"
)
deseq2_res = tidy_deseq2_table(deseq2_res)
print(deseq2_res.columns)
plot_volcano(deseq2_res, 'log2FoldChange', pcol,
            out_dir / f"Volcano_deseq2.{group_name}.png",
            title=f"DESeq2 Volcano - {group_name}")


[INFO] estimating size factors
[INFO] estimating dispersions
[INFO] gene-wise dispersion estimates
[INFO] mean-dispersion relationship
[INFO] final dispersion estimates
[INFO] fitting model and testing


Index(['feature', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue',
       'padj'],
      dtype='object')


# Old code/Sketch code

In [ ]:
import pandas as pd
import argparse 
from pathlib import Path

test_mode=True
if test_mode:
    input_metadata_tsv = Path('/home/jpereira/kraken_snake/metadata/metadata.MG1_MG2.csv')
    input_bracken_counts_tsv = Path('/home/jpereira/Results/kraken_snake/MG1_MG2/ClustDiv/bracken_combined_abundance.tsv')
    params_select_categories_dict = {
    'group1': {'Group': ['PR_R']},
    'group2': {'Group': ['PR_E']},
    'group3': {'Group': ['NS_R']},
    'group4': {'Group': ['PR_R', 'NS_R']}
    }
    params_samples_names = "LABID"
    params_metadata_subtitle = False
    output_subdivided_group_dir =  Path('/home/jpereira/Results/kraken_snake/MG1_MG2/Groups/')

output_subdivided_group_dir.mkdir(exist_ok=True)
metadata_dir = output_subdivided_group_dir / 'metadata'
counts_dir = output_subdivided_group_dir / 'counts'

metadata_dir.mkdir(exist_ok=True)
counts_dir.mkdir(exist_ok=True)

metadata_df = pd.read_csv(input_metadata_tsv, sep='\t', index_col=params_samples_names)
counts_df = pd.read_csv(input_bracken_counts_tsv, sep='\t', index_col=0)
counts_df.index.name = params_samples_names

# Discards the subtitle row if it is present 
if params_metadata_subtitle:
    metadata_df = metadata_df.loc[metadata_df.index[1::], :]
    
for group_name, filter_dict in params_select_categories_dict.items():
    metadata_group_tsv = metadata_dir / f"metadata.{group_name}.tsv"
    counts_group_tsv = counts_dir / f"counts.{group_name}.tsv"

    # Assumes only one column to filter on
    column = list(filter_dict.keys())[0]
    values = filter_dict[column]

    m_df = metadata_df[metadata_df[column].isin(values)]
    c_df = counts_df.loc[m_df.index]  # safer than iloc

    #m_df.to_csv(metadata_group_tsv, sep="\t")
    #c_df.to_csv(counts_group_tsv, sep="\t")


,Prueba,Dieta,Efficiency,Group
LABID,,,,
PR26NR063MG,26.0,R,High_Efficiency,PR_R
PR26NR042MG,26.0,R,High_Efficiency,PR_R
PR26NR026MG,26.0,R,High_Efficiency,PR_R
PR26NR007MG,26.0,R,High_Efficiency,PR_R
PR26NR001MG,26.0,R,Low_Efficiency,PR_R
PR26NR069MG,26.0,R,Low_Efficiency,PR_R
PR26NR057MG,26.0,R,Low_Efficiency,PR_R
PR26NR056MG,26.0,R,Low_Efficiency,PR_R
PR26NR044MG,26.0,R,Low_Efficiency,PR_R
